# Seed-VC 한국어 보이스 컨버전 테스트 (나의 슈퍼맨)

**목적**: Gemini Live 음성(소스)을 목표 음색(예: 엄마 목소리)으로 변환했을 때 한국어 품질이 쓸 만한지 확인.

**먼저 할 일**: 상단 메뉴 → 런타임 → 런타임 유형 변경 → **T4 GPU** 선택!

**준비물 (wav 2개)**:
1. `소스`: Gemini가 말한 음성 — 테스트 앱의 "마지막 응답 WAV 저장" 버튼으로 추출
2. `레퍼런스`: 목표 음색 5~30초 — 엄마/팀원 목소리 녹음 (조용한 곳, 또렷하게)

셀을 위에서부터 순서대로 실행하면 됨. 전체 5~10분 소요.

In [ ]:
# 1) 설치 (약 3~5분)
!git clone https://github.com/Plachtaa/seed-vc.git
%cd seed-vc
!pip install -q -r requirements.txt
print("설치 완료")

In [ ]:
# 2) 오디오 업로드
from google.colab import files

print("[1/2] 소스 오디오 업로드 (Gemini 목소리 wav)")
up = files.upload()
source_path = list(up.keys())[0]

print("[2/2] 레퍼런스 오디오 업로드 (목표 음색 5~30초 wav)")
up = files.upload()
target_path = list(up.keys())[0]

print(f"소스: {source_path} / 레퍼런스: {target_path}")

In [ ]:
# 3) 변환 실행 (체크포인트 자동 다운로드, 첫 실행만 오래 걸림)
import time
t0 = time.time()
!python inference.py --source "{source_path}" --target "{target_path}" \
  --output out --diffusion-steps 25 --inference-cfg-rate 0.7 \
  --f0-condition False --fp16 True
print(f"총 소요: {time.time()-t0:.1f}초 (모델 다운로드/로딩 포함)")

In [ ]:
# 4) 비교 청취 — 핵심 체크: 한국어 발음 뭉개짐? 음색이 레퍼런스를 닮았나? 억양 유지되나?
import glob
from IPython.display import Audio, display

print("== 원본 (Gemini) ==")
display(Audio(source_path))
print("== 레퍼런스 (목표 음색) ==")
display(Audio(target_path))
out = sorted(glob.glob("out/*.wav"))[-1]
print(f"== 변환 결과: {out} ==")
display(Audio(out))

In [ ]:
# 5) (선택) 라이브 파이프 타당성 — 1.6초 윈도우 변환 속도 측정
# 주의: CLI 호출이라 매번 모델 로딩이 포함됨. 실제 서버는 모델을 메모리에 상주시키므로
# 여기서 잰 시간보다 훨씬 빠름. 변환 자체의 로그 시간을 참고할 것.
import soundfile as sf
data, sr = sf.read(source_path)
sf.write("window16.wav", data[: int(sr * 1.6)], sr)
import time
t0 = time.time()
!python inference.py --source window16.wav --target "{target_path}" \
  --output out_win --diffusion-steps 10 --inference-cfg-rate 0.7 \
  --f0-condition False --fp16 True
print(f"1.6초 윈도우: {time.time()-t0:.1f}초 (모델 로딩 포함 — 상주 시 대폭 단축)")
from IPython.display import Audio, display
import glob
display(Audio(sorted(glob.glob('out_win/*.wav'))[-1]))

## 판정 기준

| 항목 | 통과 기준 |
|---|---|
| 한국어 발음 | 뭉개짐 없이 알아들을 수 있음 |
| 음색 유사도 | 눈 감고 들으면 레퍼런스 화자로 들림 |
| 억양 | 소스(Gemini)의 억양이 유지됨 — VC는 억양을 못 고치는 게 정상 |
| 노이즈/아티팩트 | 통화 화질이라 치면 묻힐 수준 |

**통과하면** → 다음 단계: Colab에 Seed-VC API 서버 + 터널을 띄우고 `server.mjs` VC 파이프가 ElevenLabs 대신 여기로 쏘게 교체 (무료 라이브 VC).
**미달이면** → ElevenLabs STS 경로(이미 구현됨)로 진행.